# MACRO-Mesh v2 — 완전 독립(self-contained) 실험 노트북

> **이 노트북 + 데이터 CSV(선택) 하나면 전부 실행된다.** 프로젝트의 다른 `.py`(app.services, eval_modes,
> vendored CityLearn 등)를 import하지 않는다. **미니 단지 시뮬레이터·에이전트·v2의 3모듈을 전부 아래
> 셀에 인라인 재구현**했다. 외부 의존성은 `numpy`, `pandas`, (LLM 사용 시) `openai`뿐.
>
> ⚠️ 이 노트북은 v2의 **로직(3모듈·rollout·introspector)을 충실히 재현한 *경량 standalone 버전***이다.
> 운영 코드(`app/services/citylearn_macro_mesh_v2.py`)는 LangGraph 런타임 + 실제 CityLearn env를 쓰지만,
> 여기서는 이해·검증 목적상 **가벼운 배터리/단지 시뮬레이터 + openai 호환 직접 호출**로 대체했다. 개념·흐름은 동일.

---

## 1. 문제 정의 (formal)

- **단지(district)**: N개 building. 각 building은 자기 **배터리(ESS)**(용량 `C` kWh, 출력 `P` kW)를 가진다.
- **상태(t)**: building i의 비제어 부하 `base_net_i(t)` (= 부하 − 태양광), 배터리 잔량 **SOC_i ∈ [0,1]**.
- **행동** `a_i ∈ [−1, 1]`: 배터리 충(+)/방(−). 한 step(1h)에 에너지 `a_i·P` kWh 이동.
  - SOC 갱신: `SOC_i ← clip(SOC_i + a_i·(P/C), 0, 1)` (경계에서 자동 클리핑).
  - building 실현 부하: `net_i = max(0, base_net_i + a_i·P)` (충전=부하↑, 방전=부하↓).
- **district load** `L_t = Σ_i net_i`. **목적**: `L`의 peak·합·요금·탄소↓, SOC 안전(0.2~0.9), fairness.
- **부분 관측**: 각 building agent는 자기 상태 + 단지 **요약(mean-field)** 만 본다(논문 spatiotemporal POMDP).

---

## 2. 데이터

- **입력 CSV(선택)**: ① CityLearn 스타일 `non_shiftable_load`,`solar_generation` 컬럼 → 1개 building 템플릿으로
  보고 N개로 복제(소량 스케일·노이즈) ; ② 숫자 컬럼 여러 개 → 각 컬럼 = building별 부하 시계열 ; ③ **CSV 없으면
  합성 데이터**(일주기 + 정오 PV + 노이즈, 시드 고정)로 항상 실행된다.
- 셀 3의 `DATA_CSV`에 경로를 넣거나 `None`으로 두면 합성. → **csv 한 개만 있어도, 없어도 동작.**

---

## 3. 에이전트 & 도구 구조 (전부 노트북 내부)

```
[Building Agent] × N   (LLM: openai 호환 → RUNYOUR/Sonnet, 또는 heuristic)
   └ CoProposer: 자기 building action 제안 (전략 note 주입)
              │  ThreadPoolExecutor (병렬)
              ▼
[Coordinator(결정적)] mean-field 요약 + conflict 탐지 + 2-round 협상 + merge
              ▼
[CoProposer rollout(결정적)] 후보 4종 × surrogate 비용 → 최선 채택
              ▼
   MiniDistrictEnv.step(actions) → 실현 L_t, reward
              ▼
[Coordinator(LLM)] Introspector: reward 추세 → {note, aggressiveness, discharge_bias} → 다음 step 주입
```

- LLM 호출은 `openai.OpenAI(base_url=RUNYOUR)` 로 **직접**(LangGraph·DB agent 불필요). 프롬프트(시스템/유저)는
  셀 안에 문자열로 정의 = "에이전트 생성"도 노트북에서 완결.
- 검증·수치(mean-field/conflict/rollout/metrics)는 모두 **결정적 Python**(LLM=후보생성, 검증=코드 원칙).


## 3-A. 각 모듈 내부 상세 (디테일)

> §3 다이어그램의 각 박스가 **실제로 어떤 함수·입출력으로 동작하는지**를 이 노트북의 셀 코드 기준으로 설명한다.
> (함수·변수명은 아래 코드 셀과 1:1로 일치한다.)

### 함수 ↔ 셀 빠른 지도
| 모듈(다이어그램) | 함수 | 셀 |
|---|---|---|
| 상태 제공 | `MiniDistrictEnv.observe/step` | 셀 4 |
| **CoProposer (제안)** | `llm_propose` / `heuristic_propose` / `propose_all` | 셀 5 |
| Coordinator (협상) | `mean_field` / `detect_conflicts` / `negotiate` | 셀 6 |
| CoProposer rollout (검증) | `rollout_revise` (+`_value`,`_imm`,`_soc_pen`) | 셀 7 |
| Introspector (복기) | `introspect` | 셀 8 |
| stateful 컨트롤러 | `MacroMeshV2.decide/observe` | 셀 9 |

---

### ① MiniDistrictEnv — 상태·물리 (셀 4)
- **`observe()`**: 각 building의 *배터리 적용 전* 상태를 반환 → `{building_id, net_load_kwh(=부하−PV, ≥0), battery_soc, pv}`. 이게 에이전트가 보는 **유일한 관측**.
- **`step(actions)`**: building별 `a∈[-1,1]`에 대해
  `soc_next = clip(soc + a·SOC_DELTA, 0, 1)` → 경계에 걸리면 `actual_a = (soc_next−soc)/SOC_DELTA`로 **실제 적용량만** 부하에 반영 →
  `net_i = max(0, base_net_i + actual_a·NOMINAL_POWER)`. `SOC_DELTA = P/C`(출력/용량). 충전이면 부하↑, 방전이면 부하↓.
  반환: `district L_t = Σ net_i`. (CityLearn env의 배터리 동역학을 단순화한 대체물)

---

### ② CoProposer — 자기 building action 제안 (전략 note 주입) (셀 5)
**역할**: 각 building이 *자기 배터리 action 1개*만 제안한다. 단지 전체를 보지 않고 **자기 상태 + 단지 요약(mean-field) + 전략 note**만 본다(부분 관측).

**입력** (`llm_propose(s, strategy_note, mean_field, conflicts)`):
| 입력 | 의미 | 출처 |
|---|---|---|
| `s.battery_soc / net_load_kwh / pv` | 자기 상태 | `env.observe()` |
| **`operator_strategy_note`** | **coordinator 누적 전략** | **Introspector(직전 step)** |
| `mean_field` | 이웃 제안 요약 | round1 결과(round2에서 주입) |
| `conflicts` | 충돌 목록 | round1 결과 |

**★ 전략 note 주입 경로** (이게 v2 핵심 = 시간축 기억):
```
[직전 step] introspect() → strategy["note"] = "피크 진입, 방전 강화"
        │  (MacroMeshV2.strategy 에 보관)
        ▼
[이번 step] negotiate(states, strategy_note = self.strategy["note"], ...)   # 셀 6
        ▼  propose_all(states, strategy_note, ...)                          # 셀 5
        ▼  llm_propose(...) → user JSON 의 "operator_strategy_note" 필드에 삽입
        ▼  Building LLM 이 BUILDING_SYS 의 "operator_strategy_note(상위 전략)를 반영하세요" 지시에 따라 제안에 반영
```
→ building agent는 혼자선 알 수 없는 **"지금이 피크다 / 아까 과방전으로 손해였다" 같은 전역·시간 맥락**을 이 한 줄로 전달받는다. v1엔 이 필드가 없어 매 step 백지에서 제안.

**프롬프트(실제 전송 형태)**:
- system = `BUILDING_SYS` (역할·제약·출력 JSON 규격)
- user(JSON):
```json
{"your_building":"B3","battery_soc":0.62,"net_load_kwh":3.4,
 "operator_strategy_note":"피크 진입. SOC 높은 건물 위주 방전 강화.",
 "mean_field":{"mean":-0.12,"std":0.08,...},"conflicts":[{"type":"over_discharge","n":4}]}
```
- 기대 응답: `{"action":-0.35,"mode":"discharge","confidence":0.78,"rationale":"..."}`

**후처리**: `_parse`(정규식 `\{[^{}]*\}` 중 `action` 포함 객체 파싱) → 실패 시 `None`. 이어서 `action` 을 `[-1,1]`로 클립. (LLM=후보생성, 검증·정규화=코드)

**heuristic fallback**(`USE_LLM=False` 또는 LLM 실패 시 `heuristic_propose`):
| 조건 | action |
|---|---|
| SOC<0.2 & 저부하 | +0.3 충전 |
| SOC>0.9 & 고부하(net>3) | −0.3 방전 |
| 중간 SOC & net>3 | −0.25 방전 (단 over_discharge 충돌이면 hold) |
| net<1.5 | +0.15 소량 충전 |
| else | 0 hold |

**병렬**: `propose_all` 은 `ThreadPoolExecutor(max_workers=8)`로 N개 building을 동시 호출(LLM 모드). → step당 시간 ≈ "가장 느린 building 1개" × 라운드 수.

---

### ③ Coordinator(협상) — mean-field + conflict + 2-round (셀 6)
- **`mean_field(proposals)`**: 제안들의 통계 요약 → `mean, std, mean_abs, discharge수, charge수`. 이웃 상세를 다 안 보내고 **요약만** 주고받아 확장성 확보.
- **`detect_conflicts`**: 3종 규칙 — `over_discharge`(방전 비율 ≥60%), `soc_risk`(적용 시 SOC 경계 위반), `fairness`(|action| std>0.40).
- **`negotiate(states, strategy_note, max_rounds)`**: 2라운드 루프.
  - round 0: `mean_field=None, conflicts=[]` 상태로 `propose_all` → 결과로 mf/conflicts 계산.
  - round 1: 그 **mf·conflicts를 다시 `propose_all`에 주입** → building들이 충돌을 보고 *재제안*.
  - `merged` = 마지막 라운드 제안. `consensus = 1 − min(1, std/mean_abs)` (1=합의).

---

### ④ CoProposer rollout — 실행 전 검증 (셀 7)
**역할**: 협상으로 병합된 `merged`를 *실제 적용 전에* surrogate 비용으로 평가해 **후보 4종 중 최선**을 고른다.
- 후보: `full`(병합안) / `discharge_only` / `charge_low_soc` / `hold`(∅).
- 비용 `_value = _imm(district_load, prev_L) + Σ _soc_pen(SOC')`
  - `_imm = L + PEAK_W·max(0,L−PEAK_TH) + RAMP_W·max(0,|L−Lp|−RAMP_TH)`
  - `_soc_pen`: SOC<0.3 또는 >0.9 패널티(빈/만 배터리 회피 → 미래 readiness 대리)
- `discharge_bias`로 방전쪽을 밀고, `argmin` 후보 채택 후 `× aggressiveness`로 강도 조절. → 손해 보는 제안을 가지치기(그래서 v2는 **더 적은 action으로 더 좋은 결과**).

> **CoProposer(②) ≠ rollout(④)**: ②는 *제안*(LLM/heuristic + note 반영), ④는 *검증·채택*(결정적). "제안 → 협상 → 검증" 순.

---

### ⑤ Introspector — 복기·전략 갱신 (셀 8)
- **입력**: `reward_hist`(최근 4개), `trend`(=최근−이전), 현재 `strategy`, 직전 `last_diag`. (`len<2`면 추세가 없어 skip)
- **LLM 경로**: `INTRO_SYS` + user JSON → `{note, aggressiveness∈[0.5,1.5], discharge_bias∈[-0.2,0.2]}` 파싱·클립.
- **heuristic fallback**: `trend<0`(악화) → `aggressiveness += 0.1` + note "방전 강도 상향"; 개선/안정 → 미세 감소 + "현 전략 유지".
- **출력의 행방**: 이 `{note, aggressiveness, discharge_bias}`가 **다음 step의 ②(note)와 ④(aggr/bias)에 주입** → stateful 고리 완성.

---

### ⑥ MacroMeshV2 — stateful 컨트롤러 (셀 9)
- 상태: `self.strategy = {note, aggressiveness, discharge_bias}` + `reward_hist`. (= step 간 **기억**)
- **`decide(states, prev_L)`**: `negotiate`(②③) → `rollout_revise`(④) → 채택 actions + 진단.
- **`observe(reward)`**: `reward_hist`에 추가 → `introspect`(⑤)로 `strategy` 갱신 → 다음 `decide`에 반영.

---

### 한 step 데이터 흐름 (한눈에)
```
observe() ──states──▶ negotiate(strategy.note) ──merged──▶ rollout_revise(aggr,bias) ──actions──▶ env.step()
                          │(②CoProposer가 note 반영해 제안)        │(④검증)                         │
                          └ mean_field/conflicts(③)                                                  ▼ reward
   strategy.note/aggr/bias ◀──── introspect(reward_hist) ◀───────────────────────────────── observe(reward)(⑤)
        └────────────────── 다음 step의 ②/④로 주입 (stateful) ──────────────────────────────────────┘
```

---


## 4. 의사결정 파이프라인 (stage → input → output)

| # | stage | input | output | LLM? |
|---|---|---|---|---|
| 1 | `env.observe()` | 현재 t | building별 {base_net, soc, pv} | ✗ |
| 2 | `propose_all(strategy_note)` ×N | 상태 + 전략note + mean_field/conflicts | proposal(action,conf) | ✅(병렬) |
| 3 | `mean_field` / `detect_conflicts` | proposals | 요약통계 / 충돌 | ✗ |
| 4 | round2 재제안 → `merge` | 충돌 피드백 | merged actions | ✅ |
| 5 | `rollout_revise` | merged + 상태 + prev_L + aggressiveness | revised actions | ✗ |
| 6 | `env.step` | revised actions | 실현 L_t, reward | ✗ |
| 7 | `introspect` | reward 궤적 | {note, aggressiveness, discharge_bias} | ✅(1콜) |

7의 출력 → 다음 step의 2에 주입(**stateful**: `MacroMeshV2.strategy`).

## 5. 산식 (math)

- **surrogate value** `J(plan) = imm_cost(L, L_prev) + Σ_i soc_penalty(SOC_i')`
  - `imm_cost(L,Lp) = L + PEAK_W·max(0,L−PEAK_TH) + RAMP_W·max(0,|L−Lp|−RAMP_TH)`
  - `soc_penalty`: SOC<0.3 → `(0.3−SOC)·8`, SOC>0.9 → `(SOC−0.9)·8`  (미래 readiness 대리)
- **후보** `{full, discharge_only, charge_low_soc, hold}` 중 `J` 최소 채택 → `× aggressiveness`.
- **reward** `r = −(max(L,0))^1.05`. **consensus** `= 1 − min(1, std/mean_abs)`.

## 6. 입력 매개변수 (셀 3에서 조절)

| 파라미터 | 의미 | 기본 |
|---|---|---|
| `DATA_CSV` | 데이터 CSV 경로(없으면 합성) | None |
| `N_BUILDINGS` / `HORIZON` / `START` | 건물 수 / step 수 / 시작 index | 6 / 8 / 0 |
| `INITIAL_SOC` / `CAPACITY` / `NOMINAL_POWER` | 초기 잔량 / 용량(kWh) / 출력(kW) | 0.5 / 6.4 / 5.0 |
| `USE_LLM` / `MAX_ROUNDS` | LLM 사용 / 협상 라운드 | False / 2 |
| `RUNYOUR_API_KEY`·`BASE_URL`·`MODEL` | LLM 접속(환경변수 우선) | env |

## 7. 출력 지표

- **board**: `total_consumption`, `peak`, `cumulative_reward`.
- **vs baseline(noctrl=무제어)** 비율: 소비비/피크비, 탄소=소비×carbon_rate, 요금=소비×cost_rate.
- **mesh-quality**: conflict_count, consensus, n_actions, rollout_choice.

## 8. 실행 전제

- `pip install numpy pandas` (+ LLM 쓰면 `pip install openai`).
- `USE_LLM=True`면 `RUNYOUR_API_KEY` 필요 + 유료/느림 → **먼저 `USE_LLM=False`(무료·즉시)로 검증** 권장.
- **데이터 CSV 1개 + 이 ipynb 1개로 전부 실행된다.**


In [ ]:
# [셀 1] 매개변수 — 여기만 바꿔 실험 (CSV 한 개만 있어도/없어도 동작)
import os

DATA_CSV       = None     # 예: "Building_1.csv" 또는 building별 컬럼 CSV. None이면 합성 데이터.
N_BUILDINGS    = 6
HORIZON        = 8
START          = 0
INITIAL_SOC    = 0.5
CAPACITY       = 6.4      # 배터리 용량 kWh
NOMINAL_POWER  = 5.0      # 배터리 출력 kW  (action 1.0 = 5kW)
USE_LLM        = False    # ★먼저 False(heuristic, 무료/즉시)로 검증 → 이후 True
MAX_ROUNDS     = 2

# LLM 접속(USE_LLM=True일 때만). 환경변수 우선, 없으면 아래에 직접 입력.
RUNYOUR_API_KEY = os.environ.get("RUNYOUR_API_KEY", "")          # 예: "runyour-v1-..."
RUNYOUR_BASE_URL = os.environ.get("RUNYOUR_BASE_URL", "https://api.runyour.ai/v1")
LLM_MODEL        = os.environ.get("DEFAULT_LLM_MODEL", "anthropic/claude-sonnet-4-6")

# 지표용 단가
CARBON_RATE, COST_RATE = 0.42, 0.18
print(f"config: N={N_BUILDINGS} H={HORIZON} soc={INITIAL_SOC} use_llm={USE_LLM} csv={DATA_CSV}")


In [ ]:
# [셀 2] 라이브러리 + 상수 (외부 프로젝트 파일 import 없음)
import json, math, time, random
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

SOC_DELTA   = NOMINAL_POWER / CAPACITY     # 단위 action당 SOC 변화량
SOC_LOW, SOC_HIGH = 0.20, 0.90             # 방전/충전 안전 경계
PEAK_W, RAMP_W = 2.0, 1.0                  # surrogate 가중치
RAMP_TH = 5.0
SOC_T_LOW, SOC_T_HIGH, SOC_PEN_W = 0.30, 0.90, 8.0
print("ready. SOC_DELTA=%.3f" % SOC_DELTA)


In [ ]:
# [셀 3] 데이터 로더 — CSV(있으면) 또는 합성. 반환: base[N,T], pv[N,T]
def load_series(path, n, horizon, start):
    T = start + horizon + 2
    if path and os.path.exists(path):
        df = pd.read_csv(path)
        if "non_shiftable_load" in df.columns:                 # CityLearn 스타일 1-building 템플릿 → N개 복제
            load = df["non_shiftable_load"].to_numpy(dtype=float)
            pvser = df["solar_generation"].to_numpy(dtype=float) if "solar_generation" in df.columns else np.zeros_like(load)
            need = max(T, len(load))
            load = np.resize(load, need); pvser = np.resize(pvser, need)
            base = np.stack([load * (0.8 + 0.4 * (i / max(n - 1, 1))) for i in range(n)])
            pv   = np.stack([pvser * (0.8 + 0.4 * (i / max(n - 1, 1))) for i in range(n)])
        else:                                                  # 숫자 컬럼 = building별 부하
            cols = df.select_dtypes("number").columns[:n]
            arr = df[cols].to_numpy(dtype=float).T
            base = np.resize(arr, (n, max(T, arr.shape[1]))); pv = np.zeros_like(base)
        src = f"CSV:{path}"
    else:                                                      # 합성: 일주기 + 정오 PV + 노이즈(시드 고정)
        rng = np.random.default_rng(0)
        t = np.arange(T)
        base = np.stack([2.0 + 1.2 * np.sin((t / 24.0) * 2 * np.pi - 1.0) + 0.6 * (i % 3)
                         + rng.normal(0, 0.2, T) for i in range(n)]).clip(0.2)
        pv   = np.stack([np.clip(2.5 * np.sin((t % 24 - 6) / 12 * np.pi), 0, None) * (0.6 + 0.1 * i)
                         for i in range(n)])
        src = "SYNTHETIC"
    base = np.clip(base[:, start:start + horizon + 2], 0, None)
    pv   = np.clip(pv[:, start:start + horizon + 2], 0, None)
    return base, pv, src

BASE, PV, DATA_SRC = load_series(DATA_CSV, N_BUILDINGS, HORIZON, START)
print(f"data: {DATA_SRC} | base shape {BASE.shape} | mean district base = {BASE.sum(0)[:HORIZON].mean():.1f} kW")


In [ ]:
# [셀 4] MiniDistrictEnv — 경량 배터리/단지 시뮬레이터 (CityLearn 대체, 동일 개념)
class MiniDistrictEnv:
    def __init__(self, base, pv, init_soc):
        self.base, self.pv = base, pv
        self.n = base.shape[0]
        self.reset(init_soc)
    def reset(self, init_soc=INITIAL_SOC):
        self.t = 0
        self.soc = [float(init_soc)] * self.n
        return self.observe()
    def observe(self):
        # 결정 시점: 배터리 적용 전 base net 부하(= 부하 − PV) + 현재 SOC
        net = np.clip(self.base[:, self.t] - self.pv[:, self.t], 0, None)
        return [{"building_id": f"B{i}", "net_load_kwh": float(net[i]),
                 "battery_soc": round(self.soc[i], 4), "pv_generation_kwh": float(self.pv[i, self.t])}
                for i in range(self.n)]
    def step(self, actions):
        # actions: building_id -> a∈[-1,1]. SOC 경계 클리핑 후 실제 적용량으로 부하 반영.
        net_before = np.clip(self.base[:, self.t] - self.pv[:, self.t], 0, None)
        realized = []
        for i in range(self.n):
            a = max(-1.0, min(1.0, float(actions.get(f"B{i}", 0.0))))
            soc_next = min(1.0, max(0.0, self.soc[i] + a * SOC_DELTA))
            actual_a = (soc_next - self.soc[i]) / SOC_DELTA if SOC_DELTA else 0.0
            self.soc[i] = soc_next
            realized.append(max(0.0, net_before[i] + actual_a * NOMINAL_POWER))
        self.t += 1
        return float(sum(realized)), realized

print("MiniDistrictEnv 정의 완료")


In [ ]:
# [셀 5] LLM 클라이언트(선택) + Building Agent(CoProposer) + heuristic fallback
LLM = None
if USE_LLM and RUNYOUR_API_KEY:
    from openai import OpenAI
    LLM = OpenAI(api_key=RUNYOUR_API_KEY, base_url=RUNYOUR_BASE_URL, timeout=60)
    print("LLM client ready:", LLM_MODEL)
else:
    print("LLM off → heuristic 모드 (무료/즉시)")

BUILDING_SYS = ("당신은 한 건물의 배터리(ESS) 에이전트입니다. 자기 building action ∈ [-1,1] 하나만 제안합니다. "
                "양수=충전(부하↑), 음수=방전(부하↓). SOC<0.2면 방전 금지, SOC>0.9면 충전 금지. "
                "단지 전체 부하 피크를 낮추는 것이 목표입니다. operator_strategy_note(상위 전략)를 반영하세요. "
                'final answer는 JSON: {"action": <float>, "mode": "charge|discharge|hold", "confidence": <0..1>, "rationale": "..."}')

def _parse(raw):
    import re
    for c in re.findall(r"\{[^{}]*\}", raw, re.DOTALL):
        if "action" in c:
            try: return json.loads(c)
            except Exception: pass
    return None

def heuristic_propose(s, conflicts):
    soc, net = s["battery_soc"], s["net_load_kwh"]
    over = any(c["type"] == "over_discharge" for c in conflicts)
    if soc < SOC_LOW:        a, m = (0.3, "charge") if net < 2.0 else (0.0, "hold")
    elif soc > SOC_HIGH:     a, m = (-0.3, "discharge") if net > 3.0 else (0.0, "hold")
    elif net > 3.0:          a, m = (0.0, "hold") if over else (-0.25, "discharge")
    elif net < 1.5:          a, m = (0.15, "charge")
    else:                    a, m = (0.0, "hold")
    return {"action": a, "mode": m, "confidence": 0.5, "rationale": "heuristic"}

def llm_propose(s, strategy_note, mean_field, conflicts):
    user = json.dumps({"your_building": s["building_id"], "battery_soc": s["battery_soc"],
                       "net_load_kwh": round(s["net_load_kwh"], 2),
                       "operator_strategy_note": strategy_note or "(none)",
                       "mean_field": mean_field, "conflicts": conflicts}, ensure_ascii=False)
    try:
        r = LLM.chat.completions.create(model=LLM_MODEL, temperature=0.3,
              messages=[{"role": "system", "content": BUILDING_SYS}, {"role": "user", "content": user}])
        p = _parse(r.choices[0].message.content or "")
        if p and "action" in p:
            p["action"] = max(-1.0, min(1.0, float(p["action"]))); return p
    except Exception as e:
        print("   [llm_propose fallback]", type(e).__name__)
    return heuristic_propose(s, conflicts)

def propose_all(states, strategy_note, mean_field, conflicts):
    if LLM is None:
        return [heuristic_propose(s, conflicts) for s in states]
    with ThreadPoolExecutor(max_workers=8) as ex:                 # 병렬 building 호출
        return list(ex.map(lambda s: llm_propose(s, strategy_note, mean_field, conflicts), states))

print("Building Agent(CoProposer) 정의 완료")


In [ ]:
# [셀 6] Coordinator(결정적): mean-field 요약 + conflict 탐지 + 2-round 협상
def mean_field(proposals):
    acts = [p["action"] for p in proposals]; n = len(acts) or 1
    mean = sum(acts) / n; std = (sum((a - mean) ** 2 for a in acts) / n) ** 0.5
    mean_abs = sum(abs(a) for a in acts) / n
    return {"mean": round(mean, 3), "std": round(std, 3), "mean_abs": round(mean_abs, 3),
            "discharge": sum(a < -0.05 for a in acts), "charge": sum(a > 0.05 for a in acts)}

def detect_conflicts(proposals, states):
    n = len(proposals) or 1; conf = []
    disc = [i for i, p in enumerate(proposals) if p["action"] < -0.05]
    if len(disc) / n >= 0.6:
        conf.append({"type": "over_discharge", "n": len(disc)})
    risk = [i for i, (p, s) in enumerate(zip(proposals, states))
            if not (0 <= s["battery_soc"] + p["action"] * SOC_DELTA <= 1)]
    if risk: conf.append({"type": "soc_risk", "n": len(risk)})
    mf = mean_field(proposals)
    if mf["std"] > 0.40: conf.append({"type": "fairness", "std": mf["std"]})
    return conf

def negotiate(states, strategy_note, max_rounds):
    mf, conflicts, proposals = None, [], None
    for r in range(min(max_rounds, 2)):
        proposals = propose_all(states, strategy_note, mf, conflicts)
        mf = mean_field(proposals); conflicts = detect_conflicts(proposals, states)
    merged = {s["building_id"]: p["action"] for s, p in zip(states, proposals)}
    consensus = 1 - min(1, mf["std"] / (mf["mean_abs"] + 1e-6)) if mf["mean_abs"] > 0 else 1.0
    return merged, mf, conflicts, round(consensus, 3)

print("Negotiator 정의 완료")


In [ ]:
# [셀 7] CoProposer rollout — 후보 4종 × surrogate 비용 → 최선 채택
def _district(states, act):
    return sum(max(0.0, s["net_load_kwh"] + act.get(s["building_id"], 0.0) * NOMINAL_POWER) for s in states)

def _imm(L, Lp):
    return L + PEAK_W * max(0.0, L - PEAK_TH) + RAMP_W * max(0.0, abs(L - Lp) - RAMP_TH)

def _soc_pen(soc):
    if soc < SOC_T_LOW:  return (SOC_T_LOW - soc) * SOC_PEN_W
    if soc > SOC_T_HIGH: return (soc - SOC_T_HIGH) * SOC_PEN_W
    return 0.0

def _value(states, act, Lp):
    cost = _imm(_district(states, act), Lp)
    for s in states:
        cost += _soc_pen(min(1.0, max(0.0, s["battery_soc"] + act.get(s["building_id"], 0.0) * SOC_DELTA)))
    return cost

def rollout_revise(merged, states, Lp, aggressiveness, discharge_bias):
    socs = {s["building_id"]: s["battery_soc"] for s in states}
    biased = {b: max(-1.0, min(1.0, v + (discharge_bias if v <= 0 else 0.0))) for b, v in merged.items()}
    cands = {"full": biased,
             "discharge_only": {b: v for b, v in biased.items() if v < 0},
             "charge_low_soc": {b: v for b, v in biased.items() if v > 0 and socs.get(b, 1) < SOC_T_LOW + 0.1},
             "hold": {}}
    scored = {k: _value(states, v, Lp) for k, v in cands.items()}
    best = min(scored, key=scored.get)
    chosen = {b: max(-1.0, min(1.0, v * aggressiveness)) for b, v in cands[best].items() if abs(v) >= 0.05}
    return chosen, {"rollout_choice": best, "scores": {k: round(v, 1) for k, v in scored.items()},
                    "n_actions": len(chosen)}

print("rollout_revise 정의 완료 (PEAK_TH는 셀 9에서 데이터 기준 자동 설정)")


In [ ]:
# [셀 8] Introspector — reward 추세 → 자연어 전략 + aggressiveness/discharge_bias
INTRO_SYS = ("당신은 단지 coordinator입니다. 최근 reward(0에 가까울수록 좋음)와 추세를 보고 다음 step "
             "battery 전략을 1~2문장으로 갱신하고 aggressiveness(0.5~1.5, 방전강도)와 discharge_bias(-0.2~0.2)를 "
             '조절하세요. final answer는 JSON: {"note": "...", "aggressiveness": <float>, "discharge_bias": <float>}')

def introspect(reward_hist, strategy, last_diag):
    if len(reward_hist) < 2:
        return strategy
    recent = reward_hist[-4:]; trend = recent[-1] - recent[0]
    if LLM is not None:
        try:
            import re
            user = json.dumps({"recent_rewards": [round(r, 2) for r in recent], "trend": round(trend, 2),
                               "current": strategy, "last_diag": last_diag}, ensure_ascii=False)
            r = LLM.chat.completions.create(model=LLM_MODEL, temperature=0.3,
                  messages=[{"role": "system", "content": INTRO_SYS}, {"role": "user", "content": user}])
            for c in re.findall(r"\{[^{}]*\}", r.choices[0].message.content or "", re.DOTALL):
                if "aggressiveness" in c or "note" in c:
                    d = json.loads(c)
                    return {"note": str(d.get("note", strategy["note"]))[:300],
                            "aggressiveness": max(0.5, min(1.5, float(d.get("aggressiveness", strategy["aggressiveness"])))),
                            "discharge_bias": max(-0.2, min(0.2, float(d.get("discharge_bias", strategy["discharge_bias"]))))}
        except Exception as e:
            print("   [introspect fallback]", type(e).__name__)
    # heuristic 복기: 악화하면 더 공격적으로, 개선이면 유지
    aggr = strategy["aggressiveness"] + (0.1 if trend < 0 else -0.02)
    note = ("reward 악화 → 방전 강도 상향" if trend < 0 else "reward 개선/안정 → 현 전략 유지")
    return {"note": note, "aggressiveness": max(0.5, min(1.5, aggr)), "discharge_bias": strategy["discharge_bias"]}

print("introspect 정의 완료")


In [ ]:
# [셀 9] MacroMeshV2 컨트롤러(stateful) + PEAK_TH 자동설정
PEAK_TH = float(BASE.sum(0)[START:START + HORIZON].max()) * 0.9   # 데이터 기준 peak 임계
print("PEAK_TH = %.1f kW" % PEAK_TH)

class MacroMeshV2:
    def __init__(self, max_rounds):
        self.max_rounds = max_rounds
        self.strategy = {"note": "", "aggressiveness": 1.0, "discharge_bias": 0.0}
        self.reward_hist = []
        self.last_diag = {}
    def decide(self, states, prev_L):
        merged, mf, conflicts, consensus = negotiate(states, self.strategy["note"], self.max_rounds)   # ②③④
        chosen, rdiag = rollout_revise(merged, states, prev_L,                                          # ⑤
                                       self.strategy["aggressiveness"], self.strategy["discharge_bias"])
        diag = {"rollout_choice": rdiag["rollout_choice"], "n_actions": rdiag["n_actions"],
                "consensus": consensus, "conflict_count": len(conflicts), "scores": rdiag["scores"]}
        self.last_diag = diag
        return chosen, diag
    def observe(self, reward):
        self.reward_hist.append(reward)                                                                 # ⑦
        self.strategy = introspect(self.reward_hist, self.strategy, self.last_diag)

print("MacroMeshV2 정의 완료")


In [ ]:
# [셀 10] 실행 루프 — 매 step: 상태→협상→rollout→적용→복기 (터미널 출력)
def run(controller, init_soc=INITIAL_SOC, verbose=True):
    env = MiniDistrictEnv(BASE, PV, init_soc)
    states = env.observe(); hist, loads = [], []
    if verbose:
        print(f"{'step':>4} {'time':>6} {'L_after':>8} {'rollout':>15} {'act':>4} {'cons':>5} {'conf':>4} {'aggr':>5}")
        print("-" * 70)
    for step in range(HORIZON):
        L_before = sum(s["net_load_kwh"] for s in states); prev = hist[-1] if hist else L_before
        hist.append(L_before)
        t0 = time.time()
        actions, diag = controller.decide(states, prev)
        L_after, _ = env.step(actions)
        loads.append(L_after)
        controller.observe(-(max(L_after, 0.0) ** 1.05))
        states = env.observe()
        if verbose:
            print(f"{step+1:>4} {time.time()-t0:>5.1f}s {L_after:>8.1f} {diag['rollout_choice']:>15} "
                  f"{diag['n_actions']:>4} {diag['consensus']:>5.2f} {diag['conflict_count']:>4} "
                  f"{controller.strategy['aggressiveness']:>5.2f}")
            if controller.strategy.get("note"):
                print("       └", controller.strategy["note"][:110])
    return loads

print(">>> v2 실행")
CTL_V2 = MacroMeshV2(MAX_ROUNDS)      # 컨트롤러를 변수로 보관(누적 전략 확인용)
loads_v2 = run(CTL_V2)


In [ ]:
# [셀 11] 결과 — board 지표 + 무제어(noctrl) baseline 대비 비율
class NoCtrl:
    strategy = {"note": "", "aggressiveness": 1.0, "discharge_bias": 0.0}
    def decide(self, states, prev): return {}, {"rollout_choice": "noctrl", "n_actions": 0, "consensus": 1.0, "conflict_count": 0}
    def observe(self, r): pass

loads_base = run(NoCtrl(), verbose=False)

def board(loads):
    return {"total_consumption_kwh": round(sum(loads), 1), "peak_load_kw": round(max(loads), 1),
            "cumulative_reward": round(sum(-(max(L, 0) ** 1.05) for L in loads), 1)}

bv, bb = board(loads_v2), board(loads_base)
cons_ratio = bv["total_consumption_kwh"] / bb["total_consumption_kwh"]
peak_ratio = bv["peak_load_kw"] / bb["peak_load_kw"]
print("noctrl(baseline):", bb)
print("macro_mesh_v2   :", bv)
print()
print(f"소비비(v2/base) = {cons_ratio:.3f}  | 피크비 = {peak_ratio:.3f}  (<1이면 개선)")
print(f"탄소(v2) = {bv['total_consumption_kwh']*CARBON_RATE:.1f} kgCO2 | 요금(v2) = ${bv['total_consumption_kwh']*COST_RATE:.1f}")
print(f"v2 최종 누적 전략: aggr={CTL_V2.strategy['aggressiveness']:.2f} "
      f"discharge_bias={CTL_V2.strategy['discharge_bias']:.2f} note='{CTL_V2.strategy['note'][:80]}'")


In [ ]:
# [셀 12] (선택) rollout 내부 — 가상 plan의 후보별 surrogate 비용 비교
env = MiniDistrictEnv(BASE, PV, INITIAL_SOC); states = env.observe()
demo = {s["building_id"]: -0.3 for s in states}          # 전원 0.3 방전 가정
chosen, rdiag = rollout_revise(demo, states, sum(s["net_load_kwh"] for s in states), 1.0, 0.0)
print("후보별 surrogate 비용:", rdiag["scores"])
print("선택:", rdiag["rollout_choice"], "| 채택 action 수:", rdiag["n_actions"])
print("해석: 비용이 가장 낮은 후보가 채택된다. 방전이 부하를 낮추면 discharge_only가 유리해짐.")
